In [0]:
import re
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import to_timestamp, current_timestamp
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime


# Padronização dos nomes dos schemas e volumes
CATALOG = "vcredit_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")


CATALOG: vcredit_medallion
BRONZE_SCHEMA: vcredit_medallion.bronze
SILVER_SCHEMA: vcredit_medallion.silver
GOLD_SCHEMA: vcredit_medallion.gold
LANDING_PATH: /Volumes/vcredit_medallion/bronze/landing


## vcredit_medallion.bronze.tb_base_motivos

preenchi os 3 nulls que tinha em categoria usando a categorização descrita no pdf do case
mudei o formato do id pra int e do ingestion_datetime para timestamp (vou mudar o nome dessa coluna na bronze dps, mas aí eu arrumo tudo)

In [0]:
df_bronze_base_motivos = spark.table(f"{BRONZE_SCHEMA}.tb_base_motivos")

display(df_bronze_base_motivos)

df_silver_base_motivos = (
    df_bronze_base_motivos
    # trim em TODAS as colunas
    .select(*[F.trim(F.col(c)).alias(c) for c in df_bronze_base_motivos.columns])
    # cast do id_motivo para int
    .withColumn("id_motivo", F.col("id_motivo").try_cast("int"))
    # cast do ingestion_datetime para timestamp
     .withColumn("ingestion_datetime", F.col("ingestion_datetime").cast("timestamp"))
    # preencher categoria NULL
    .withColumn(
        "categoria",
        F.when(F.col("categoria").isNull() & (F.col("id_motivo") == 14), F.lit("Cadastral"))
         .when(F.col("categoria").isNull() & (F.col("id_motivo") == 16), F.lit("Cadastral"))
         .when(F.col("categoria").isNull() & (F.col("id_motivo") == 12), F.lit("Cartao"))
         .otherwise(F.col("categoria"))
    )
    # padronizar criticidade
    .withColumn(
        "criticidade",
        F.when(F.lower(F.col("criticidade")) == "alta",  F.lit("alta"))
         .when(F.lower(F.col("criticidade")) == "media", F.lit("media"))
         .when(F.lower(F.col("criticidade")) == "média", F.lit("media"))
         .when(F.lower(F.col("criticidade")) == "baixa", F.lit("baixa"))
         .otherwise(F.col("criticidade"))
    )
)

display(df_silver_base_motivos)

df_silver_base_motivos.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.tb_base_motivos")

id_motivo,nome_motivo,categoria,criticidade,ingestion_datetime
11,Bloqueio de cartão,Cartao,Alta,2026-10-24T16:24:14.103Z
7,Compra não autorizada,Cartao,alta,2026-10-24T16:24:14.103Z
13,Dúvidas gerais sobre programa de pontos,Beneficios,baixa,2026-10-24T16:24:14.103Z
14,Transferência de agência,null,Media,2026-10-24T16:24:14.103Z
9,Problema com aplicativo,Atendimento,Media,2026-10-24T16:24:14.103Z
8,Consulta de contrato,Financeiro,Media,2026-10-24T16:24:14.103Z
15,Renegociação de dívida,Financeiro,alta,2026-10-24T16:24:14.103Z
16,Cancelamento de conta,null,Alta,2026-10-24T16:24:14.103Z
4,"Alteração de dados cadastrais (vencimento da fatura, telefone, email)",Cadastral,baixa,2026-10-24T16:24:14.103Z
3,Contestação de fatura,Financeiro,alta,2026-10-24T16:24:14.103Z


id_motivo,nome_motivo,categoria,criticidade,ingestion_datetime
11,Bloqueio de cartão,Cartao,alta,2026-10-24T16:24:14.103Z
7,Compra não autorizada,Cartao,alta,2026-10-24T16:24:14.103Z
13,Dúvidas gerais sobre programa de pontos,Beneficios,baixa,2026-10-24T16:24:14.103Z
14,Transferência de agência,Cadastral,media,2026-10-24T16:24:14.103Z
9,Problema com aplicativo,Atendimento,media,2026-10-24T16:24:14.103Z
8,Consulta de contrato,Financeiro,media,2026-10-24T16:24:14.103Z
15,Renegociação de dívida,Financeiro,alta,2026-10-24T16:24:14.103Z
16,Cancelamento de conta,Cadastral,alta,2026-10-24T16:24:14.103Z
4,"Alteração de dados cadastrais (vencimento da fatura, telefone, email)",Cadastral,baixa,2026-10-24T16:24:14.103Z
3,Contestação de fatura,Financeiro,alta,2026-10-24T16:24:14.103Z


##  vcredit_medallion.bronze.tb_canais

so mudei os tipos e passei um trim

In [0]:
df_bronze_canais = spark.table(f"{BRONZE_SCHEMA}.tb_canais")

display(df_bronze_canais)

df_silver_canais = (
    df_bronze_canais
    # trim em TODAS as colunas
    .select(*[F.trim(F.col(c)).alias(c) for c in df_bronze_canais.columns])
    # cast do id_canal para int
    .withColumn("id_canal", F.col("id_canal").try_cast("int"))
    # cast do ingestion_datetime para timestamp
     .withColumn("ingestion_datetime", F.col("ingestion_datetime").cast("timestamp"))
)

display(df_silver_canais)

df_silver_canais.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.tb_canais")

id_canal,nome_canal,tipo_canal,status_canal,ingestion_datetime
1,URA,Autosservico,ativo,2026-10-24T16:24:20.952Z
2,CHATBOT,Autosservico,ativo,2026-10-24T16:24:20.952Z
3,ATENDIMENTO INICIAL,Humano,ativo,2026-10-24T16:24:20.952Z
4,ATENDIMENTO ESPECIALIZADO,Humano,ativo,2026-10-24T16:24:20.952Z


id_canal,nome_canal,tipo_canal,status_canal,ingestion_datetime
1,URA,Autosservico,ativo,2026-10-24T16:24:20.952Z
2,CHATBOT,Autosservico,ativo,2026-10-24T16:24:20.952Z
3,ATENDIMENTO INICIAL,Humano,ativo,2026-10-24T16:24:20.952Z
4,ATENDIMENTO ESPECIALIZADO,Humano,ativo,2026-10-24T16:24:20.952Z
